# Home Credit EDA (씬파일러 가중치 근거)

Give Me Some Credit에는 실제 씬파일러가 없어서 시뮬레이터가 씬파일러를 만든다.
어떤 고객을 더 높은 확률로 씬파일러로 뽑을지(가중치)의 근거를, 실제 씬파일러가 있는 Home Credit 데이터에서 찾는다.

## 1. 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.config import INCOME_INVALID_MAX, RANDOM_STATE
from src.data.loader import load_give_me_some_credit, load_home_credit
from src.data.preprocessor import drop_invalid_rows, split_data
from src.data.simulator import (
    cap_thin_features, generate_alternative_data, gmsc_thin_features, home_credit_thin_features,
)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
application, bureau = load_home_credit()

In [ ]:
def show_thin_rate(data: pd.DataFrame, group: pd.Series | pd.Categorical, title: str) -> None:
    """그룹별 인원과 씬파일러 비율을 표와 막대그래프로 보여준다."""
    table = data.groupby(group, observed=True)["is_thin"].agg(["count", "mean"])
    table.columns = ["인원", "씬파일러 비율(%)"]
    table["씬파일러 비율(%)"] = (table["씬파일러 비율(%)"] * 100).round(1)
    display(table)

    x = range(len(table))
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.bar(x, table["씬파일러 비율(%)"], color="steelblue")
    ax.axhline(THIN_RATE, color="red", linestyle="--", label=f"전체 씬파일러 비율 {THIN_RATE:.1f}%")
    for i, (n, rate) in enumerate(zip(table["인원"], table["씬파일러 비율(%)"])):
        ax.text(i, rate, f"{n:,}명", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(table.index.astype(str), rotation=30, ha="right")
    ax.set_title(title)
    ax.set_ylabel("씬파일러 비율(%)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. 씬파일러 정의

과제 기준은 "금융 연체 관련 컬럼 2개 이상 결측이거나 신용카드 거래 이력 12개월 미만"이다.
Home Credit에서 연체 정보는 CB 기록(`bureau`)에만 있다. CB 기록이 없는 고객은 연체 정보가 전부 없으므로 첫 번째 조건에 해당한다.

**CB 기록이 한 건도 없는 고객을 씬파일러로 정의한다.** Give Me Some Credit에서 씬파일러를 만드는 방식(연체 컬럼 3개를 결측으로 만듦)과도 같은 조건이다.

In [ ]:
is_thin = (~application["SK_ID_CURR"].isin(bureau["SK_ID_CURR"])).astype(int).rename("is_thin")
application = pd.concat([application, is_thin], axis=1)
THIN_RATE = application["is_thin"].mean() * 100
print(f"전체 {len(application):,}명 중 씬파일러 {application['is_thin'].sum():,}명 ({THIN_RATE:.1f}%)")

table = application.groupby("is_thin")["TARGET"].agg(["count", "mean"])
table.index = ["CB 기록 있음", "씬파일러"]
table.columns = ["인원", "부도율(%)"]
table["부도율(%)"] = (table["부도율(%)"] * 100).round(2)
table

**결론**
- 씬파일러는 307,511명 중 44,020명(14.3%)이다.
- 씬파일러의 부도율(10.1%)이 CB 기록이 있는 고객(7.7%)보다 높다. 금융 이력이 없어 기존 방식으로 평가하기 어려운 고객이라는 과제 설명과 맞는다.

## 3. 컬럼 후보 고르기

가중치에 쓸 컬럼은 두 조건을 모두 만족해야 한다.
1. **Give Me Some Credit에도 있다.** 없으면 Home Credit에서 구한 가중치를 적용할 수 없다.
2. **CB 기록에서 만든 값이 아니다.** CB 기록에서 만든 값은 씬파일러가 정의상 0이거나 결측이라 순환 논리가 된다 (4절).

| 컬럼 묶음 | ① Give Me Some Credit에 있음 | ② CB 기록에서 만들지 않음 | 결과 |
| --- | --- | --- | --- |
| `DAYS_BIRTH` (나이) | ✅ `age` | ✅ | **후보** |
| `AMT_INCOME_TOTAL` (소득) | ✅ `MonthlyIncome` | ✅ | **후보** |
| `CNT_CHILDREN` (자녀 수) | ✅ `NumberOfDependents` | ✅ | **후보** |
| `AMT_ANNUITY` ÷ `AMT_INCOME_TOTAL` (상환액 ÷ 소득) | ✅ `DebtRatio` | ✅ | **후보** |
| `bureau`, `bureau_balance` (대출 수, 연체 횟수, 한도 사용률 등) | ✅ | ❌ | 제외 |
| `AMT_REQ_CREDIT_BUREAU_*` (CB 조회 횟수 6개) | ❌ | ❌ | 제외 |
| `previous_application`, `POS_CASH_balance`, `credit_card_balance`, `installments_payments` (Home Credit 내부 이력) | ❌ | ✅ | 제외 |
| 나머지 `application` 컬럼 (학력, 주거 형태, 혼인 상태, 외부 점수 등) | ❌ | ✅ | 제외 |

CB 조회 횟수가 CB 기록과 이어져 있는지 확인한다.

In [ ]:
missing = application.groupby("is_thin")["AMT_REQ_CREDIT_BUREAU_YEAR"].apply(lambda s: s.isna().mean() * 100).round(1)
missing.index = ["CB 기록 있음", "씬파일러"]
missing.to_frame("CB 조회 횟수 결측률(%)")

**결론**: CB 조회 횟수는 씬파일러의 94.3%가 결측이고 CB 기록이 있는 고객은 결측이 없다. CB 기록에 딸린 값이므로 제외한다.

## 4. CB 기록 컬럼을 빼는 이유

Give Me Some Credit의 씬파일러 가중치는 지금 `NumberOfOpenCreditLinesAndLoans`(대출 수)로 정한다. Home Credit에서 대출 수에 해당하는 값(CB 기록 중 사용 중인 대출 건수)별로 씬파일러 비율을 센다.

In [ ]:
open_lines = bureau[bureau["CREDIT_ACTIVE"] == "Active"].groupby("SK_ID_CURR").size()
open_lines = application["SK_ID_CURR"].map(open_lines).fillna(0)

group = open_lines.map(lambda n: "0개" if n == 0 else "1개 이상").rename("사용 중인 대출 수")
show_thin_rate(application, group, "사용 중인 대출 수별 씬파일러 비율")

**결론: CB 기록에서 만든 컬럼은 가중치에 쓰지 않는다.**
- 대출 1개 이상은 씬파일러 비율이 0%다. 고객의 경향이 아니라, 대출 수를 CB 기록으로 세기 때문에 정의상 0%가 나온 것이다.
- 이 비율을 가중치로 쓰면 "대출 1개 이상은 절대 씬파일러가 아니다"라는 규칙이 된다. Give Me Some Credit에서 대출 0개인 고객은 1.26%(1,888명)뿐이라 씬파일러를 30% 만들 수 없다.
- "대출이 적을수록 씬파일러에 가깝다"는 방향은 맞지만, 얼마나 가까운지(크기)는 Home Credit으로 알 수 없다.

## 5. 후보 4개의 경향

두 데이터에서 같은 의미가 되도록 바꾼다. 변환 함수는 `src/data/simulator.py`에 있고, 시뮬레이터도 같은 함수로 Give Me Some Credit을 바꾼다.

| 공통 컬럼 | Home Credit | Give Me Some Credit | 변환 |
| --- | --- | --- | --- |
| `age` | `DAYS_BIRTH` | `age` | 음수 일수 ÷ (−365.25) → 나이(년) |
| `income_pct` | `AMT_INCOME_TOTAL` | `MonthlyIncome` | 데이터 안에서의 백분위 (화폐 단위가 다름) |
| `dependents` | `CNT_CHILDREN` | `NumberOfDependents` | 그대로 |
| `debt_ratio_pct` | `AMT_ANNUITY` ÷ `AMT_INCOME_TOTAL` | `DebtRatio` | 데이터 안에서의 백분위 (정의가 다름, 아래 참고) |

- `DebtRatio`는 두 데이터의 정의가 다르다. Give Me Some Credit은 "모든 부채 상환액 + 생활비 ÷ 월소득"(소득 정상 고객 중앙값 0.29), Home Credit은 "이번 대출 상환액 ÷ 소득"(중앙값 0.16)이다. 값을 그대로 비교할 수 없어 백분위로 바꾼다.
- Give Me Some Credit에서 소득이 결측이거나 0~10인 고객은 소득과 `DebtRatio`를 백분위 0.5(중앙값)로 둔다.

In [ ]:
features = home_credit_thin_features(application)
features["is_thin"] = application["is_thin"]
features.describe().round(3)

### 5-1. 나이

In [ ]:
group = pd.cut(features["age"], bins=[20, 25, 30, 35, 40, 45, 50, 55, 60, 70], include_lowest=True)
show_thin_rate(features, group, "나이 구간별 씬파일러 비율")

- 25세 이하는 27.6%로 전체 평균(14.3%)의 약 2배다. 25~30세도 17.2%로 높다.
- 30세 이후는 11.7~14.8%로 큰 차이가 없다.
- 과제의 씬파일러 예시(사회초년생)와 맞는다.

### 5-2. 소득 백분위

In [ ]:
group = pd.cut(features["income_pct"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1], include_lowest=True)
show_thin_rate(features, group, "소득 백분위 구간별 씬파일러 비율")

- 소득이 낮을수록 씬파일러가 많다. 하위 20% 구간 19.2% → 상위 20% 구간 10.9%로 꾸준히 줄어든다.
- 구간별 인원이 고르지 않은 것은 같은 소득(예: 135,000)인 고객이 많아서 백분위가 겹치기 때문이다.

### 5-3. 부양가족(자녀 수)

In [ ]:
group = features["dependents"].clip(upper=3).map({0: "0명", 1: "1명", 2: "2명", 3: "3명 이상"})
group = pd.Categorical(group, categories=["0명", "1명", "2명", "3명 이상"], ordered=True)
show_thin_rate(features, group, "자녀 수별 씬파일러 비율")

- 0명 14.8%, 1명 이상 12.5~13.6%로 차이가 작다.
- Home Credit은 자녀 수, Give Me Some Credit은 부양가족 수(배우자 등 포함)라 의미도 완전히 같지는 않다.

### 5-4. 상환액 ÷ 소득 백분위

In [ ]:
group = pd.cut(features["debt_ratio_pct"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1], include_lowest=True)
show_thin_rate(features, group, "상환액 ÷ 소득 백분위 구간별 씬파일러 비율")

- 소득 대비 상환 부담이 클수록 씬파일러가 많다. 하위 20% 구간 12.0% → 상위 20% 구간 18.2%로 꾸준히 늘어난다.

## 6. 결론

**가중치에 쓸 컬럼: 4개 모두 사용한다.**

| 공통 컬럼 | 경향 | 차이 |
| --- | --- | --- |
| `age` | 젊을수록 씬파일러가 많다 (25세 이하 27.6%) | 큼 |
| `income_pct` | 소득이 낮을수록 많다 (19.2% → 10.9%) | 중간 |
| `debt_ratio_pct` | 상환 부담이 클수록 많다 (12.0% → 18.2%) | 중간 |
| `dependents` | 0명이 조금 많다 (14.8% vs 12.5~13.6%) | 작음 |

- 컬럼마다 영향의 크기는 다음 단계의 로지스틱 회귀 계수로 정한다. 차이가 작은 `dependents`는 계수도 작게 나올 것이다.
- 나이의 경향은 직선이 아니다. 30세까지는 크게 줄고 그 뒤로는 비슷하다. 로지스틱 회귀에 나이를 그대로 넣으면 이 모양을 직선으로 펴서 반영하므로, 다음 단계에서 나이를 넣는 방식을 확인한다.
- Home Credit 고객의 나이는 20~69세이고, Give Me Some Credit은 21~109세다. 70세 이상은 Home Credit에 없는 나이라 계수를 그대로 쓰면 범위 밖의 값을 추정하게 된다.

## 7. 가중치 계산 (로지스틱 회귀)

4개 컬럼으로 씬파일러 여부를 맞히는 로지스틱 회귀를 학습하고, 계수를 가중치로 쓴다. 로지스틱 회귀는 나이와 소득처럼 서로 관련된 컬럼의 효과를 나눠서 계수에 반영하므로, 같은 효과가 두 번 들어가지 않는다.

- 부양가족은 3명 이상을 3으로 자른다. 4명 이상은 두 데이터 모두 드물어 계수가 불안정해진다.

### 7-1. 나이를 넣는 방식 비교

6절의 나이 문제(직선 모양이 아님, 70세 이상 없음)를 해결하는 두 방식을 5-Fold 교차검증 AUC로 비교한다.

| 방식 | 내용 | 나이 계수 개수 |
| --- | --- | --- |
| 30세에서 자르기 | 30세 이상은 모두 30으로 본다 | 1개 |
| 구간 | 25세 이하 / 25~30세 / 30세 초과로 나눈다 (30세 초과가 기준) | 2개 |

**선택 기준**: AUC가 높은 쪽을 고른다. 차이가 0.005 미만이면 계수가 적은 30세에서 자르기를 고른다.

In [ ]:
y = features["is_thin"]
base = features.drop(columns=["is_thin", "age"]).assign(dependents=features["dependents"].clip(upper=3))
candidates = {
    "30세에서 자르기": base.assign(age=features["age"].clip(upper=30)),
    "구간": base.assign(
        age_le25=(features["age"] <= 25).astype(int),
        age_25_30=((features["age"] > 25) & (features["age"] <= 30)).astype(int),
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
result = {}
for name, X in candidates.items():
    scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=cv, scoring="roc_auc")
    result[name] = {"AUC 평균": scores.mean(), "AUC 표준편차": scores.std()}
pd.DataFrame(result).T.round(4)

**결론: 30세에서 자르기를 고른다.**
- AUC가 0.5929로 구간 방식(0.5923)보다 조금 높고, 계수도 1개로 적다.
- 30세 이상을 모두 30으로 보므로, Give Me Some Credit의 70세 이상 고객도 Home Credit 범위 밖의 값을 추정하지 않는다.
- 상한값(나이 30, 부양가족 3)은 `config.py`의 `THIN_AGE_CAP`, `THIN_DEPENDENTS_CAP`에 있고, `simulator.py`의 `cap_thin_features()`가 자른다.
- AUC가 0.59라는 것은 4개 컬럼이 씬파일러를 약하게만 가려낸다는 뜻이다. 가중치도 고객 간 차이가 크지 않게 나온다 (검증 단계에서 다시 확인한다).

### 7-2. 최종 학습

Home Credit 전체로 학습한다.

In [ ]:
X = cap_thin_features(features.drop(columns="is_thin"))
model = LogisticRegression(max_iter=1000).fit(X, y)

### 7-3. config에 적을 값

계수의 뜻: 값이 1단위 오를 때 씬파일러 오즈(씬파일러일 확률 ÷ 아닐 확률)가 몇 배가 되는지를 `오즈 배수`로 함께 보여준다. 백분위 컬럼은 0~1 범위라 0.1 단위로 본다.

In [ ]:
unit = {"age": 1, "income_pct": 0.1, "dependents": 1, "debt_ratio_pct": 0.1}
coefs = pd.Series(model.coef_[0], index=X.columns)
pd.DataFrame({
    "계수": coefs.round(4),
    "단위": [unit[c] for c in coefs.index],
    "단위당 오즈 배수": np.exp(coefs * pd.Series(unit)).round(3),
})

In [ ]:
print("THIN_FILER_COEFS = {" + ", ".join(f'"{c}": {v:.4f}' for c, v in coefs.items()) + "}")

**결론: 위에 출력된 줄을 `config.py`의 씬파일러 설정에 적는다.**
- 절편은 적지 않는다. Home Credit의 씬파일러 비율(14.3%)에 맞춘 값이라, 시뮬레이터가 `thin_filer_ratio`에 맞춰 절편을 다시 구한다.
- `age`: 30세까지 1살 많을수록 씬파일러 오즈가 약 0.87배가 된다. 20세는 30세보다 약 3.8배다.
- `income_pct`: 소득 백분위가 0.1 오를 때마다 약 0.94배가 된다. 소득 최하위는 최상위보다 약 1.9배다.
- `debt_ratio_pct`: 상환 부담 백분위가 0.1 오를 때마다 약 1.04배가 된다. 최상위는 최하위보다 약 1.5배다.
- `dependents`: 1명 많을수록 약 0.93배로, 영향이 가장 작다.

## 8. 검증

### 8-1. Home Credit AUC

7-1에서 고른 모델의 5-Fold 교차검증 AUC는 **0.5929**다 (0.5 = 무작위, 1.0 = 완벽).
- 4개 컬럼은 씬파일러를 약하게만 가려낸다. 실제 씬파일러 여부는 이 4개보다 다른 이유에 더 크게 좌우된다는 뜻이다.
- 그래서 시뮬레이터가 만드는 씬파일러도 나이·소득에 따라 조금씩만 달라진다. Home Credit의 실제 모습을 따른 결과라 문제로 보지 않는다.

### 8-2. Give Me Some Credit에서 씬파일러 생성

시뮬레이터 기본값(`thin_filer_ratio` = 0.3)으로 씬파일러를 만든다.

In [ ]:
gmsc = drop_invalid_rows(load_give_me_some_credit())
gmsc_train, _, _ = split_data(gmsc)   # 시뮬레이터의 신용 점수 모델은 Train으로만 학습한다
gmsc = generate_alternative_data(gmsc, gmsc_train.index)
gmsc_features = gmsc_thin_features(gmsc).assign(is_thin=gmsc["is_thin_filer"])
valid_income = gmsc["MonthlyIncome"] > INCOME_INVALID_MAX   # 소득이 정상인 고객

### 8-3. 구간별 비교

**비교 방법**
- 두 데이터는 전체 씬파일러 비율이 14.3%와 30%로 달라서 구간별 비율을 그대로 비교할 수 없다.
- 그래서 **오즈 배수**(구간의 씬파일러 오즈 ÷ 전체 오즈)로 비교한다. 1보다 크면 평균보다 씬파일러가 많은 구간이다. 시뮬레이터는 계수를 그대로 두고 절편만 바꾸므로, 이 값이 Home Credit과 비슷하게 나와야 한다.

**통과 기준**
- 모든 구간에서 오즈 배수의 방향(1보다 큰지 작은지)이 두 데이터에서 같으면 통과로 본다.
- 30세 초과 나이 구간은 예외로 본다. 나이를 30세에서 잘라 이 구간의 작은 굴곡을 일부러 반영하지 않았기 때문이다.
- 소득과 상환 부담은 Give Me Some Credit에서 소득이 결측이거나 0~10인 고객을 빼고 비교한다. 이 고객들은 백분위가 0.5로 고정되어 가운데 구간에 몰리기 때문이다.

In [ ]:
def thin_odds_table(data: pd.DataFrame, group: pd.Series | pd.Categorical) -> pd.DataFrame:
    """그룹별 인원, 씬파일러 비율, 오즈 배수(그룹의 씬파일러 오즈 ÷ 전체 오즈)를 구한다."""
    table = data.groupby(group, observed=True)["is_thin"].agg(["count", "mean"])
    overall = data["is_thin"].mean()
    table["오즈 배수"] = (table["mean"] / (1 - table["mean"])) / (overall / (1 - overall))
    table["mean"] *= 100
    return table.rename(columns={"count": "인원", "mean": "씬파일러 비율(%)"})


def compare_thin_odds(
    hc_data: pd.DataFrame, hc_group: pd.Series | pd.Categorical,
    gmsc_data: pd.DataFrame, gmsc_group: pd.Series | pd.Categorical, title: str,
) -> None:
    """두 데이터의 구간별 오즈 배수를 표와 막대그래프로 비교한다."""
    table = pd.concat({
        "Home Credit": thin_odds_table(hc_data, hc_group),
        "Give Me Some Credit": thin_odds_table(gmsc_data, gmsc_group),
    }, axis=1)
    hc_odds, gmsc_odds = table[("Home Credit", "오즈 배수")], table[("Give Me Some Credit", "오즈 배수")]
    same = np.sign(np.log(hc_odds)) == np.sign(np.log(gmsc_odds))
    table[("비교", "방향")] = np.where(hc_odds.isna(), "-", np.where(same, "같음", "다름"))
    display(table.round(3))

    x = np.arange(len(table))
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.bar(x - 0.2, hc_odds, width=0.4, label="Home Credit", color="steelblue")
    ax.bar(x + 0.2, gmsc_odds, width=0.4, label="Give Me Some Credit (생성)", color="darkorange")
    ax.axhline(1, color="red", linestyle="--", label="평균 (오즈 배수 1)")
    ax.set_xticks(x)
    ax.set_xticklabels(table.index.astype(str), rotation=30, ha="right")
    ax.set_title(title)
    ax.set_ylabel("오즈 배수")
    ax.legend()
    plt.tight_layout()
    plt.show()

#### 나이

In [ ]:
bins = [20, 25, 30, 35, 40, 45, 50, 55, 60, 70, 110]
compare_thin_odds(
    features, pd.cut(features["age"], bins=bins, include_lowest=True),
    gmsc_features, pd.cut(gmsc_features["age"], bins=bins, include_lowest=True),
    "나이 구간별 씬파일러 오즈 배수",
)

- 25세 이하(2.28 → 2.67)와 25~30세(1.25 → 1.35)는 두 데이터 모두 평균보다 씬파일러가 많다. Give Me Some Credit 쪽이 조금 더 강하게 나왔다.
- 30~35세와 60~70세는 방향이 다르지만, 오즈 배수가 두 데이터 모두 1 근처(0.94~1.04)이고 예외로 정한 30세 초과 구간이다.
- 70세 이상은 Home Credit에 없는 나이라 비교할 수 없다. 30세로 잘랐기 때문에 30세 이후 고객과 비슷하게(0.99) 나왔다.

#### 소득 백분위

In [ ]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]
compare_thin_odds(
    features, pd.cut(features["income_pct"], bins=bins, include_lowest=True),
    gmsc_features[valid_income], pd.cut(gmsc_features.loc[valid_income, "income_pct"], bins=bins, include_lowest=True),
    "소득 백분위 구간별 씬파일러 오즈 배수",
)

- 모든 구간에서 방향이 같고 크기도 거의 같다 (하위 20% 1.42 → 1.45, 상위 20% 0.73 → 0.72).

#### 부양가족

In [ ]:
compare_thin_odds(
    features, features["dependents"].clip(upper=3),
    gmsc_features, gmsc_features["dependents"].clip(upper=3),
    "부양가족 수별 씬파일러 오즈 배수 (3 = 3명 이상)",
)

- 모든 구간에서 방향이 같다. 3명 이상은 Give Me Some Credit 쪽이 조금 더 낮다 (0.94 → 0.78).

#### 상환 부담 백분위

In [ ]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]
compare_thin_odds(
    features, pd.cut(features["debt_ratio_pct"], bins=bins, include_lowest=True),
    gmsc_features[valid_income], pd.cut(gmsc_features.loc[valid_income, "debt_ratio_pct"], bins=bins, include_lowest=True),
    "상환 부담 백분위 구간별 씬파일러 오즈 배수",
)

- 모든 구간에서 방향이 같다. Give Me Some Credit 쪽이 조금 약하다 (하위 20% 0.82 → 0.92, 상위 20% 1.33 → 1.27).

### 8-4. 결론

**통과: 예외 구간을 빼고 모든 구간에서 방향이 같다.**

| 컬럼 | 방향 | 크기 |
| --- | --- | --- |
| 나이 | 30세 이하 모두 같음 (30세 초과는 예외) | Give Me Some Credit 쪽이 조금 강함 |
| 소득 백분위 | 모두 같음 | 거의 같음 |
| 부양가족 | 모두 같음 | 3명 이상에서 Give Me Some Credit 쪽이 조금 낮음 |
| 상환 부담 백분위 | 모두 같음 | Give Me Some Credit 쪽이 조금 약함 |

- 시뮬레이터가 Home Credit에서 구한 관계(젊을수록, 소득이 낮을수록, 상환 부담이 클수록 씬파일러가 많음)를 Give Me Some Credit에 옮겼다.
- 크기가 조금씩 다른 것은 두 데이터의 고객 구성(나이 분포, 소득과 나이의 관계 등)이 다르기 때문으로 보인다. 같은 계수라도 구간 안에 어떤 고객이 섞여 있는지에 따라 구간의 오즈 배수가 달라진다.